<a href="https://colab.research.google.com/github/DarshiniMahesh/Chatbots-AI/blob/main/Interior_Designer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install Required Libraries
!pip install gradio requests Pillow -q

print("✅ All libraries installed!")

✅ All libraries installed!


In [ ]:
# Step 2: Set Your API Token
HF_TOKEN = "Token"  # ← paste here

if HF_TOKEN == "hf_paste_your_token_here":
    print("⚠️  Please replace with your actual token!")
else:
    print("✅ Token is set! Ready to go!")

✅ Token is set! Ready to go!


In [ ]:
# Step 3: Import Libraries
import gradio as gr
import requests
import io
import base64
import time
import numpy as np
from PIL import Image

print("✅ All imports done!")
print(f"Gradio version: {gr.__version__}")

✅ All imports done!
Gradio version: 5.50.0


In [ ]:
# Step 4: Design Styles with Prompts
DESIGN_STYLES = {
    "🏙️ Modern Minimalist": {
        "prompt": "modern minimalist interior design, clean lines, white walls, "
                  "natural light, neutral palette, scandinavian furniture, "
                  "open space, uncluttered, professional interior photography, "
                  "8k, photorealistic",
        "negative": "cluttered, dark, old, dirty, low quality"
    },
    "🌿 Bohemian": {
        "prompt": "bohemian interior design, warm earthy tones, rattan furniture, "
                  "indoor plants, colorful textiles, macrame wall hangings, "
                  "cozy atmosphere, natural materials, professional photography, "
                  "8k, photorealistic",
        "negative": "sterile, cold, minimal, modern, low quality"
    },
    "⚙️ Industrial": {
        "prompt": "industrial interior design, exposed brick walls, metal pipes, "
                  "concrete floors, dark tones, Edison bulb lighting, steel furniture, "
                  "urban loft style, raw materials, professional photography, "
                  "8k, photorealistic",
        "negative": "soft, pastel, cozy, rustic, low quality"
    },
    "👑 Luxury Classic": {
        "prompt": "luxury classic interior design, gold accents, marble floors, "
                  "crystal chandeliers, velvet furniture, ornate details, rich colors, "
                  "elegant drapes, high-end decor, professional photography, "
                  "8k, photorealistic",
        "negative": "cheap, minimalist, plain, simple, low quality"
    },
    "🌊 Coastal Breeze": {
        "prompt": "coastal beach house interior design, light blue and white palette, "
                  "natural wood, sea glass decor, linen curtains, rattan accents, "
                  "bright airy spaces, professional photography, 8k, photorealistic",
        "negative": "dark, urban, industrial, heavy, low quality"
    },
    "🌸 Japanese Zen": {
        "prompt": "japanese zen interior design, tatami mats, shoji screens, "
                  "bamboo elements, neutral earth tones, bonsai plants, "
                  "minimalist furniture, soft natural light, peaceful atmosphere, "
                  "professional photography, 8k, photorealistic",
        "negative": "cluttered, colorful, western, ornate, low quality"
    }
}

print(f"✅ {len(DESIGN_STYLES)} design styles loaded!")
for style in DESIGN_STYLES:
    print(f"  {style}")

✅ 6 design styles loaded!
  🏙️ Modern Minimalist
  🌿 Bohemian
  ⚙️ Industrial
  👑 Luxury Classic
  🌊 Coastal Breeze
  🌸 Japanese Zen


In [ ]:
# Step 5: Image Preprocessing
def preprocess_image(image, max_size=512):
    """
    Resize image for Stable Diffusion.
    SD needs: dimensions divisible by 8, ideally 512x512.
    """
    # Convert to RGB (removes transparency/alpha channel)
    image = image.convert("RGB")

    # Resize keeping aspect ratio
    image.thumbnail((max_size, max_size), Image.LANCZOS)

    # Make width & height divisible by 8
    w, h = image.size
    w = (w // 8) * 8
    h = (h // 8) * 8
    image = image.resize((w, h), Image.LANCZOS)

    return image


def image_to_base64(image):
    """
    Convert PIL Image → base64 string.
    APIs can't receive raw image files, so we encode them as text.
    """
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=95)
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


print("✅ Preprocessing functions ready!")

# Quick test
test_img = Image.new("RGB", (800, 600), color="white")
processed = preprocess_image(test_img)
print(f"Original size : 800 x 600")
print(f"Processed size: {processed.size[0]} x {processed.size[1]}")
print(f"Divisible by 8: {processed.size[0] % 8 == 0} ✅")

✅ Preprocessing functions ready!
Original size : 800 x 600
Processed size: 512 x 384
Divisible by 8: True ✅


In [ ]:
# Step 6: Main Generation Function
API_URL = "https://router.huggingface.co/hf-inference/models/stabilityai/stable-diffusion-xl-base-1.0"

def generate_interior_design(room_image, style_name, strength):
    """
    Sends room photo + style prompt to Stable Diffusion API.

    strength: float between 0.3 and 0.95
        0.3 = subtle changes (keeps most of original)
        0.95 = dramatic transformation (almost new room)
    """

    if room_image is None:
        return None, "⚠️ Please upload a room image first!"

    try:
        # ── 1. Preprocess image ──
        processed = preprocess_image(room_image)
        img_b64   = image_to_base64(processed)
        print(f"📸 Image ready: {processed.size}")

        # ── 2. Get style prompts ──
        style_data      = DESIGN_STYLES[style_name]
        prompt          = style_data["prompt"]
        negative_prompt = style_data["negative"]
        print(f"🎨 Style: {style_name}")
        print(f"📝 Prompt: {prompt[:60]}...")

        # ── 3. Build API request ──
        headers = {
            "Authorization": f"Bearer {HF_TOKEN}",
            "Content-Type": "application/json"
        }

        payload = {
            "inputs": prompt,
            "parameters": {
                "negative_prompt":    negative_prompt,
                "num_inference_steps": 30,   # more steps = better quality (but slower)
                "guidance_scale":      7.5,  # how closely to follow the prompt
                "strength":            strength,
                "image":               img_b64
            }
        }

        # ── 4. Call API with retry logic ──
        # HF free tier sometimes needs model warmup (20-30 seconds)
        print("🚀 Calling Hugging Face API...")

        for attempt in range(3):
            response = requests.post(
                API_URL,
                headers=headers,
                json=payload,
                timeout=120
            )

            if response.status_code == 503:
                # Model is loading — wait and retry
                print(f"⏳ Model loading... waiting 20s (attempt {attempt+1}/3)")
                time.sleep(20)
                continue

            if response.status_code == 200:
                print("✅ API responded successfully!")
                break

            # Any other error
            return None, f"❌ API Error {response.status_code}: {response.text[:200]}"

        if response.status_code != 200:
            return None, "❌ Model still loading. Wait 30 seconds and try again!"

        # ── 5. Convert response → PIL Image ──
        result_image = Image.open(io.BytesIO(response.content))
        print(f"🏡 Result image size: {result_image.size}")

        return result_image, f"✅ Successfully redesigned as {style_name}!"

    except requests.exceptions.Timeout:
        return None, "⏱️ Timed out! HF free tier is slow sometimes. Try again!"
    except Exception as e:
        return None, f"❌ Unexpected error: {str(e)}"


print("✅ Generation function ready!")

✅ Generation function ready!


In [ ]:
# Step 7: Test API Connection
def test_api():
    headers = {
        "Authorization": f"Bearer {HF_TOKEN}",
        "Content-Type": "application/json"
    }
    test_payload = {"inputs": "a white room"}
    response = requests.post(API_URL, headers=headers, json=test_payload, timeout=60)

    if response.status_code == 200:
        print("✅ API Connected & Working!")
        print(f"   Response size: {len(response.content)} bytes")
        print("   Model is live — ready to generate!")
    elif response.status_code == 401:
        print("❌ Invalid token! Double-check your HF_TOKEN")
    elif response.status_code == 503:
        print("⚠️ Model warming up (normal) — proceed anyway!")
    elif response.status_code == 410:
        print("❌ Model URL still wrong — try Option 2 below")
    else:
        print(f"⚠️ Status {response.status_code}: {response.text[:200]}")

test_api()

✅ API Connected & Working!
   Response size: 41720 bytes
   Model is live — ready to generate!


In [ ]:
# Step 8: Build Gradio UI

# ── Style descriptions ──
STYLE_INFO = {
    "🏙️ Modern Minimalist": "Clean lines, white spaces, clutter-free. Perfect for small apartments.",
    "🌿 Bohemian":          "Warm earthy tones, plants, and eclectic cozy decor. For free spirits.",
    "⚙️ Industrial":        "Exposed brick, raw metal, and urban loft feel. Bold and edgy.",
    "👑 Luxury Classic":    "Gold accents, marble, chandeliers. Pure opulence and elegance.",
    "🌊 Coastal Breeze":    "Light blues, natural wood, and breezy vibes. Bring the beach home!",
    "🌸 Japanese Zen":      "Bamboo, shoji screens, peaceful minimalism. Find your inner calm."
}

# ── Wrapper function ──
def redesign_room(room_image, style_choice, strength):
    if room_image is None:
        return None, "⚠️ Please upload a room photo first!"
    if isinstance(room_image, np.ndarray):
        room_image = Image.fromarray(room_image)
    return generate_interior_design(room_image, style_choice, float(strength))

def update_style_info(style):
    return f"🎨 **{style}** — {STYLE_INFO.get(style, '')}"

# ── Build App ──
with gr.Blocks(title="🏠 AI Interior Designer") as app:

    gr.Markdown("# 🏠 AI Interior Designer\nTransform any room into your dream space using AI ✨")

    with gr.Row():

        # Left — Inputs
        with gr.Column():
            input_image = gr.Image(type="pil", label="📸 Upload Room Photo", height=300)

            style_dropdown = gr.Dropdown(
                choices=list(DESIGN_STYLES.keys()),
                value="🏙️ Modern Minimalist",
                label="🎨 Design Style"
            )

            style_desc = gr.Markdown("Select a style to see its description")

            strength_slider = gr.Slider(
                minimum=0.3, maximum=0.95, value=0.6, step=0.05,
                label="🎚️ Transformation Strength",
                info="0.3 = subtle  |  0.6 = balanced  |  0.95 = dramatic"
            )

            generate_btn = gr.Button("✨ Redesign My Room!", variant="primary", size="lg")
            status_box   = gr.Textbox(label="⚡ Status", interactive=False)

        # Right — Output
        with gr.Column():
            output_image = gr.Image(label="🏡 Redesigned Room", height=300)

            gr.Markdown("""
            **💡 Tips:**
            - Use well-lit, clear room photos
            - Start with strength **0.6** then adjust
            - First run may take 20–30 seconds (model warmup)
            """)

    # ── Wire up events ──
    style_dropdown.change(fn=update_style_info, inputs=style_dropdown, outputs=style_desc)

    generate_btn.click(
        fn=redesign_room,
        inputs=[input_image, style_dropdown, strength_slider],
        outputs=[output_image, status_box]
    )

# ── Launch ──
app.launch(share=True, debug=True)